In [ ]:
# Run first on Google Colab
!pip install qutip -q

# §3 Quantum Phase Estimation

**Course:** Introductory Quantum Computing — Summer School

## Learning objectives
- Implement QPE as a matrix circuit and observe the output distribution
- Understand the precision theorem experimentally (success probability vs clock qubits)
- Verify the O(1/2^t) eigenphase error bound numerically


In [ ]:
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt
from fractions import Fraction
from math import gcd

print(f"QuTiP {qt.__version__}")

# ── Helpers carried over from Notebook 1 ─────────────────────────────────────
def qft_matrix(n):
    N = 2**n
    omega = np.exp(2j * np.pi / N)
    F = np.array([[omega**(j*k) / np.sqrt(N) for k in range(N)] for j in range(N)])
    return qt.Qobj(F, dims=[[2]*n, [2]*n])

def apply_to_qubit(gate, n, qubit):
    ops = [qt.qeye(2)] * n; ops[qubit] = gate
    return qt.tensor(ops)

def controlled_u(U, n, ctrl, tgt):
    P0 = qt.basis(2,0)*qt.basis(2,0).dag()
    P1 = qt.basis(2,1)*qt.basis(2,1).dag()
    ops0 = [qt.qeye(2)]*n; ops0[ctrl] = P0
    ops1 = [qt.qeye(2)]*n; ops1[ctrl] = P1; ops1[tgt] = U
    return qt.tensor(ops0) + qt.tensor(ops1)

def swap_gate(n, i, j):
    N = 2**n; mat = np.zeros((N,N),dtype=complex)
    for k in range(N):
        bits = list(format(k,f'0{n}b')); bits[i],bits[j]=bits[j],bits[i]
        mat[int(''.join(bits),2),k]=1.0
    return qt.Qobj(mat, dims=[[2]*n,[2]*n])

def qft_circuit(n):
    U = qt.tensor([qt.qeye(2)]*n); H1 = qt.gates.hadamard_transform(1)
    for i in range(n):
        U = apply_to_qubit(H1,n,i)*U
        for j in range(i+1,n):
            m=j-i+1; Rm=qt.Qobj(np.diag([1.0,np.exp(2j*np.pi/2**m)]))
            U = controlled_u(Rm,n,j,i)*U
    for i in range(n//2): U = swap_gate(n,i,n-1-i)*U
    return U


---
## Part 1: Quantum Phase Estimation (QPE)

### 3.1 The algorithm

**Problem:** Given unitary $U$ with eigenpair $(e^{2\pi i\varphi}, |u\rangle)$, estimate the eigenphase $\varphi \in [0,1)$.

**Classical analogy:** Observe the signal $s_k = e^{2\pi ik\varphi}$ and take its DFT — the peak sits at $j = \lfloor N\varphi\rceil$.
QPE does exactly this *in superposition* using **phase kickback** + **inverse QFT**.

**Circuit (Algorithm 3.1):**
1. Initialise $|0^t\rangle|u\rangle$ (clock ⊗ system)
2. Apply $\mathsf{H}^{\otimes t}$ to clock: $\frac{1}{\sqrt{2^t}}\sum_j |j\rangle|u\rangle$
3. For $j=0,\ldots,t-1$: apply controlled-$U^{2^j}$ (control = clock qubit $j$)
   — phase kickback gives: $\frac{1}{\sqrt{2^t}}\sum_j e^{2\pi ij\varphi}|j\rangle|u\rangle$
4. Apply $\mathsf{QFT}_t^\dagger$ to clock
5. Measure clock → output $\tilde{\varphi} = k/2^t \approx \varphi$

**Precision theorem:** With $t = n + \lceil\log_2(2 + 1/(2\varepsilon))\rceil$ clock qubits, $|\tilde{\varphi} - \varphi| \leq 2^{-n}$ with probability $\geq 1-\varepsilon$.


In [ ]:
def qpe(U_system, eigenstate, t_clock):
    """
    Run QPE and return the full output state (clock ⊗ system).

    Parameters
    ----------
    U_system  : QuTiP Qobj  — the unitary whose eigenphase we estimate
    eigenstate: QuTiP Qobj  — eigenvector of U_system
    t_clock   : int         — number of clock qubits

    Returns
    -------
    psi_out : QuTiP Qobj with dims [[2]*t_clock + sys_dims[0], ...]
    """
    sys_dim   = U_system.shape[0]
    sys_n     = int(np.round(np.log2(sys_dim)))
    sys_dims  = [[2]*sys_n, [2]*sys_n] if sys_n > 0 else [[sys_dim],[sys_dim]]

    zero = qt.basis(2, 0)
    H1   = qt.gates.hadamard_transform(1)
    I_sys = qt.qeye(sys_dim);  I_sys.dims = sys_dims

    # ── Step 1: initialise |0^t>|u> ──────────────────────────────────────────
    clock_zero = qt.tensor([zero]*t_clock)
    psi = qt.tensor(clock_zero, eigenstate)

    # ── Step 2: Hadamard on all clock qubits ──────────────────────────────────
    Ht = qt.gates.hadamard_transform(t_clock)
    H_full = qt.tensor(Ht, I_sys)
    # Fix dims after tensor
    total_n = t_clock + sys_n
    H_full.dims = [[2]*total_n, [2]*total_n]
    psi = H_full * psi

    # ── Step 3: controlled-U^{2^(t-1-j)} for j=0..t-1 ───────────────────────
    for j in range(t_clock):
        # Qubit j (MSB=0 in QuTiP tensor order) controls U^{2^(t-1-j)}
        power = t_clock - 1 - j
        U_pow = U_system.copy()
        for _ in range(power):
            U_pow = U_pow * U_pow

        # Embed: identity on clock, U^{2^j} on system
        # clock qubit j is control; system is target
        # Build as |0><0|_j ⊗ I_rest + |1><1|_j ⊗ U_j
        P0 = qt.basis(2,0) * qt.basis(2,0).dag()
        P1 = qt.basis(2,1) * qt.basis(2,1).dag()
        I_clock = qt.tensor([qt.qeye(2)]*(t_clock-1)) if t_clock > 1 else qt.qeye(1)

        # Build clock projectors
        ops0 = [qt.qeye(2)]*t_clock;  ops0[j] = P0
        ops1 = [qt.qeye(2)]*t_clock;  ops1[j] = P1
        clock_P0 = qt.tensor(ops0)
        clock_P1 = qt.tensor(ops1)

        gate = qt.tensor(clock_P0, I_sys) + qt.tensor(clock_P1, U_pow)
        gate.dims = [[2]*total_n, [2]*total_n]
        psi = gate * psi

    # ── Step 4: QFT† on clock ─────────────────────────────────────────────────
    QFT_dag = qft_circuit(t_clock).dag()
    QFT_full = qt.tensor(QFT_dag, I_sys)
    QFT_full.dims = [[2]*total_n, [2]*total_n]
    psi = QFT_full * psi

    return psi

def measure_clock(psi_out, t_clock, sys_n, n_shots=2048):
    """
    Simulate measurement of the clock register and return histogram.
    Returns outcome counts dict {k: count} and probabilities array.
    """
    total_n = t_clock + sys_n
    N_total = 2**total_n
    N_clock = 2**t_clock
    N_sys   = 2**sys_n

    amps = psi_out.full().flatten()
    # Sum over system states for each clock value
    probs_clock = np.zeros(N_clock)
    for k in range(N_clock):
        for s in range(N_sys):
            idx = k * N_sys + s
            probs_clock[k] += abs(amps[idx])**2

    outcomes = np.random.choice(N_clock, size=n_shots, p=probs_clock)
    counts = {k: int(np.sum(outcomes == k)) for k in range(N_clock) if np.sum(outcomes == k) > 0}
    return counts, probs_clock

print("QPE helpers defined.")


In [ ]:
# ── Simple test: U = phase gate, eigenphase = φ ───────────────────────────────
phi_true = 3/8   # exact t=3 bit phase → QPE should give outcome 3 deterministically
U_test = qt.Qobj(np.diag([1.0, np.exp(2j*np.pi*phi_true)]))  # eigenvalue e^{2πiφ}|1>
eigvec = qt.basis(2, 1)   # |1> is the eigenstate

for t in [3, 4, 5]:
    psi_out = qpe(U_test, eigvec, t)
    counts, probs = measure_clock(psi_out, t, 1, n_shots=4096)
    # Most probable outcome
    best_k = max(counts, key=counts.get)
    phi_est = best_k / 2**t
    print(f"t={t}: best outcome k={best_k:3d}, φ̃={phi_est:.5f}, error={abs(phi_est-phi_true):.2e}")

# ── Plot probability distribution for t=4 ────────────────────────────────────
t = 4
psi_out4 = qpe(U_test, eigvec, t)
_, probs4 = measure_clock(psi_out4, t, 1)

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(2**t), probs4)
ax.axvline(phi_true * 2**t, color='red', linestyle='--', label=f'exact: k={phi_true*2**t:.1f}')
ax.set_xlabel('clock outcome k')
ax.set_ylabel('probability')
ax.set_title(f'QPE output distribution (φ={phi_true}, t={t} clock qubits)')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ── Precision experiment: φ=1/3 (not exactly representable) ──────────────────
phi_true = 1/3
U_13 = qt.Qobj(np.diag([1.0, np.exp(2j*np.pi*phi_true)]))
eigvec = qt.basis(2, 1)

t_range = range(2, 8)
success_probs = []

for t in t_range:
    psi_out = qpe(U_13, eigvec, t)
    _, probs = measure_clock(psi_out, t, 1)
    # Success = outcome within 1 of nearest integer to 2^t * phi
    nearest = int(np.round(phi_true * 2**t))
    success = sum(probs[k % 2**t] for k in [nearest-1, nearest, nearest+1])
    success_probs.append(success)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(list(t_range), success_probs, 'o-')
ax.axhline(4/np.pi**2, color='gray', linestyle='--', label='4/π² ≈ 0.405 (theory min)')
ax.set_xlabel('clock qubits t')
ax.set_ylabel('success probability')
ax.set_title('QPE precision: φ=1/3, success = |k/2^t − φ| ≤ 1/2^t')
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.show()


### Exercise 3.1 — QPE with $\varphi = 1/3$, $t = 3$

**(a)** Analytically compute the probability that QPE outputs $k=1$ (i.e., $\tilde{\varphi}=1/8$).

**(b)** Compute the probability of output $k=0$, and verify the total success probability (outputs within $1/2^3$ of $\varphi$) exceeds $1/2$.

*Hint:* The amplitude of outcome $k$ after $\mathsf{QFT}^\dagger_t$ is
$\alpha_k = \frac{1}{2^t}\sum_{j=0}^{2^t-1} e^{2\pi ij(\varphi - k/2^t)}$.


In [ ]:
# YOUR CODE HERE

# (a) Compute P(k=1) analytically for φ=1/3, t=3
# phi = 1/3, t = 3, N = 2^t = 8
# alpha_k = (1/8) * sum_{j=0}^{7} exp(2*pi*i*j*(1/3 - k/8))

# (b) Compute P(k=0) and check P(k=0) + P(k=1) + nearby outcomes > 1/2

# Then verify numerically:
phi = 1/3;  t = 3;  N = 2**t
_, probs = measure_clock(qpe(qt.Qobj(np.diag([1.0, np.exp(2j*np.pi*phi)])),
                             qt.basis(2,1), t), t, 1, n_shots=1)
# probs[k] gives exact probability of each clock outcome


In [ ]:
#@title Solution — Exercise 3.1  {display-mode: "form"}
phi = 1/3;  t = 3;  N = 2**t

def alpha(k, phi, t):
    """Amplitude of clock outcome k."""
    N = 2**t
    return sum(np.exp(2j*np.pi*j*(phi - k/N)) for j in range(N)) / N

# (a) P(k=1)
a1 = alpha(1, phi, t)
print(f"alpha(k=1) = {a1:.6f}")
print(f"P(k=1)     = {abs(a1)**2:.6f}")

# (b) P(k=0) and nearby
for k in range(N):
    p = abs(alpha(k, phi, t))**2
    print(f"  P(k={k}) = {p:.6f}   (φ̃={k/N:.4f})")

nearest = int(np.round(phi * N))   # = 0 (nearest integer to 8/3=2.67 is 3)
# Actually nearest integer to 8*1/3 = 2.667 is 3
print(f"\nNearest k to 2^t*φ = {phi*N:.4f} is k={nearest}")
success = sum(abs(alpha(k, phi, t))**2 for k in [nearest-1, nearest, nearest+1])
print(f"P(success) = P(k ∈ {{{nearest-1},{nearest},{nearest+1}}}) = {success:.4f}")
print(f"Theoretical lower bound 4/π² ≈ {4/np.pi**2:.4f}")


### Exercise 3.2 — Verifying the precision theorem

The precision theorem states: to estimate $\varphi$ with error $|\tilde{\varphi}-\varphi| \leq 2^{-n}$
with probability $\geq 1-\varepsilon$, it suffices to use
$$t = n + \left\lceil\log_2\!\left(2 + \tfrac{1}{2\varepsilon}\right)\right\rceil \text{ clock qubits.}$$

Take $\varphi = 1/3$ (not exactly representable in binary) and $n = 3$ (so target error $\leq 1/8$).

**(a)** For $\varepsilon \in \{0.5, 0.1, 0.05\}$ compute the required $t$ from the formula above.

**(b)** For each $t$, use `alpha(k, phi, t)` to compute the exact success probability
$P_{\text{succ}} = \sum_{|k/2^t - \varphi| \leq 2^{-n}} |\alpha_k|^2$
and verify it meets the $1-\varepsilon$ guarantee.

**(c)** Show that using $t - 1$ clock qubits *fails* the guarantee for at least one $\varepsilon$.

In [ ]:
# YOUR CODE HERE

import math

phi = 1/3
n   = 3          # target precision: error <= 2^{-n} = 1/8
epsilons = [0.5, 0.1, 0.05]

# (a) Required clock qubits from precision theorem
# t = n + ceil(log2(2 + 1/(2*eps)))
# t_required = ...

# (b) Success probability at t_required: sum |alpha_k|^2 over k with |k/2^t - phi| <= 2^{-n}
# Use alpha(k, phi, t) defined in the solution cell above
# p_succ = ...

# (c) Check t_required - 1
# p_succ_minus1 = ...

In [ ]:
#@title Solution — Exercise 3.2  {display-mode: "form"}
import math

phi = 1/3
n   = 3
epsilons = [0.5, 0.1, 0.05]

def success_prob(phi, t, n):
    """Exact success probability: sum |alpha_k|^2 over k with |k/2^t - phi| <= 2^{-n}."""
    N = 2**t
    threshold = 1 / 2**n
    return sum(
        abs(alpha(k, phi, t))**2
        for k in range(N)
        if abs(k/N - phi) <= threshold or abs(k/N - phi - 1) <= threshold
    )

print(f"phi = {phi:.6f},  target precision 2^(-n) = 2^(-{n}) = {1/2**n:.4f}")
print()
print(f"{'eps':>6}  {'t (formula)':>12}  {'P_succ(t)':>12}  {'1-eps':>8}  {'guarantee met?':>15}  {'P_succ(t-1)':>13}  {'fails at t-1?':>14}")
print('-' * 92)
for eps in epsilons:
    t = n + math.ceil(math.log2(2 + 1/(2*eps)))
    p  = success_prob(phi, t,   n)
    p1 = success_prob(phi, t-1, n)
    print(f"{eps:>6.2f}  {t:>12d}  {p:>12.6f}  {1-eps:>8.4f}  {'yes' if p >= 1-eps else 'NO':>15}  {p1:>13.6f}  {'yes' if p1 < 1-eps else 'no':>14}")

---
## Summary

| Topic | Key result |
|-------|-----------|
| QPE algorithm | Hadamard on clock + controlled-$U^{2^j}$ (phase kickback) + $\mathsf{QFT}^\dagger$; reads out eigenphase $\varphi$ |
| Precision | $t$ clock qubits give resolution $2^{-t}$; $t = n + \lceil\log_2(2+\tfrac{1}{2\varepsilon})\rceil$ ensures error $\leq 2^{-n}$ with prob $\geq 1-\varepsilon$ |
| Amplitude | $\alpha_k = \frac{1}{2^t}\sum_{j=0}^{2^t-1}e^{2\pi ij(\varphi - k/2^t)}$; Dirichlet kernel centred on $k^* = \lfloor 2^t\varphi\rceil$ |
| Bit ordering | QuTiP tensor order is MSB-first: qubit $j$ controls $U^{2^{t-1-j}}$; clock integer $k$ has bit $(k\gg(t-1-j))\&1$ at position $j$ |

**Next:** Notebook 4 uses QPE as a subroutine in Shor's factoring algorithm (§4).